<a href="https://colab.research.google.com/github/Przemo75/git_training/blob/master/HAKI_sortowanie_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd

# === 1. Wczytanie plików ===
new_file = '/content/_sort_new.csv'
old_file = '/content/_sort_old.csv'

df_new = pd.read_csv(new_file)
df_old = pd.read_csv(old_file)

columns = ['MARKA', 'MODEL', 'ROCZNIK', 'RODZAJ NADWOZIA']

# === 2. Tworzenie maski zmian ===
diff_mask = pd.DataFrame(False, index=df_new.index, columns=columns)

for col in columns:
    old_values = set(df_old[col].dropna().unique())
    diff_mask[col] = ~df_new[col].isin(old_values)

print("Nowe wartości:", diff_mask.sum().sum())

Nowe wartości: 2364


In [13]:
output_file = '/content/nowe_wartosci.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_new.to_excel(writer, index=False, sheet_name='Wynik')

    workbook = writer.book
    worksheet = writer.sheets['Wynik']

    from openpyxl.styles import PatternFill
    red_fill = PatternFill(start_color='FF0000', end_color='FF0000', fill_type='solid')

    for row in range(len(df_new)):
        for col in range(len(df_new.columns)):
            if diff_mask.iat[row, col]:
                worksheet.cell(row=row+2, column=col+1).fill = red_fill

from google.colab import files
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
# Słownik, który będzie przechowywał listy nowych wartości dla każdej kolumny
new_vals_dict = {}

for col in columns:
    # Pobieramy unikalne wartości z df_new tam, gdzie maska wskazała zmianę
    unique_news = df_new.loc[diff_mask[col], col].dropna().unique()
    # Zapisujemy jako Series, co pozwoli na złożenie tabeli o różnej długości kolumn
    new_vals_dict[col] = pd.Series(unique_news)

# Tworzymy DataFrame z list (kolumny zostaną wyrównane do najdłuższej)
df_new_values = pd.DataFrame(new_vals_dict)

# Opcjonalnie: zamiana NaN na pusty ciąg znaków, żeby w Excelu było czysto
df_new_values = df_new_values.fillna("")

print("Tabela nowych wartości przygotowana.")

Tabela nowych wartości przygotowana.


In [15]:
summary = {
    "nowe_marki": df_new.loc[diff_mask["MARKA"], "MARKA"].nunique(),
    "nowe_modele": df_new.loc[diff_mask["MODEL"], "MODEL"].nunique(),
    "nowe_roczniki": df_new.loc[diff_mask["ROCZNIK"], "ROCZNIK"].nunique(),
    "nowe_nadwozia": df_new.loc[diff_mask["RODZAJ NADWOZIA"], "RODZAJ NADWOZIA"].nunique(),
}

df_summary = pd.DataFrame(list(summary.items()), columns=["kategoria", "liczba"])

In [16]:
output_final = "porownanie_koncowe.xlsx"

with pd.ExcelWriter(output_final, engine="openpyxl") as writer:
    # 1. Arkusz z pełnymi danymi i czerwonym podświetleniem (logika z kroku 2)
    df_new.to_excel(writer, sheet_name="NEW_DATA", index=False)

    # --- Tutaj dodajemy logikę kolorowania dla arkusza NEW_DATA ---
    worksheet = writer.sheets['NEW_DATA']
    from openpyxl.styles import PatternFill
    red_fill = PatternFill(start_color='FF0000', end_color='FF0000', fill_type='solid')

    for row in range(len(df_new)):
        for col_idx, col_name in enumerate(df_new.columns):
            if col_name in columns: # sprawdzamy tylko kolumny zdefiniowane w liście columns
                if diff_mask.iat[row, columns.index(col_name)]:
                    worksheet.cell(row=row+2, column=col_idx+1).fill = red_fill
    # -------------------------------------------------------------

    # 2. Arkusz z listą unikalnych nowych wartości (4 kolumny)
    df_new_values.to_excel(writer, sheet_name="NOWE_WARTOSCI", index=False)

    # 3. Arkusz z podsumowaniem liczbowym
    df_summary.to_excel(writer, sheet_name="PODSUMOWANIE", index=False)

from google.colab import files
files.download(output_final)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>